# T0.3 — Acquisition-start order census

**Goal:** identify which A0–B2 sessions have HARP starting before or after SLAP2, estimate the possible source-cycle offset, and stage a safe reprocessing queue. This is a read-only audit, **not proof of one-frame absolute alignment**.

**Sources:** `VIPSessionRegistry`, DMD1 `.meta` MAT-file creation headers and cycle parameters, `behavior/launcher_metadata/launcher.log` (Bonsai workflow start and HARP recording-folder timestamp), and existing HARP DI3 QC when available.

**Interpretation:** Positive `HARP − SLAP2` means SLAP2 appears to start first; explicit timing bounds prevent an unsupported hard decision.

In [ ]:
%matplotlib notebook
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

## Settings
Run `scripts/run_t03_start_order_audit.py --summary-xlsx ... --output-dir ...` first. The optional CSV is a **manifest file-presence census only**, not a timing audit. Change `EVIDENCE_CSV` below.

In [ ]:
EVIDENCE_CSV = Path(r"C:\Users\andrew.shelton\Desktop\T03_start_order\t03_start_order_evidence.csv")
OUTPUT_DIR = Path(r"./T03_start_order/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not EVIDENCE_CSV.exists():
    raise FileNotFoundError(f"Run scripts/run_t03_start_order_audit.py first: {EVIDENCE_CSV}")
df = pd.read_csv(EVIDENCE_CSV)
print(f"{len(df)} sessions in audit evidence table")
display(df[["session_id", "order_classification", "estimated_harp_minus_source_s", "assumed_timing_bound_s", "reprocess_disposition"]].head(20))

## Figure 1. Session counts by acquisition-order evidence
**Data:** local `.meta` timing and launcher-log/HARP-folder timing, resolved through the registry. **Meaning:** this counts classifications with a deliberately conservative uncertainty bound, not physical synchronization passes.

In [ ]:
order = ["slap2_first", "harp_first", "ambiguous_overlap", "missing_timing_evidence"]
counts = df.order_classification.fillna("missing_timing_evidence").value_counts()
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.bar(np.arange(4), [counts.get(x, 0) for x in order])
ax.set_xticks(np.arange(4), ["SLAP2 first", "HARP first", "Ambiguous", "Missing"], rotation=12, ha="right")
ax.set_ylabel("Sessions"); ax.set_title("Acquisition order: A0–B2 registry sessions")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"01_order_coverage.png", dpi=250)
plt.show()

## Figure 2. Estimated start-time differences with bounds
**Data:** source `.meta` MAT-file creation time versus the HARP directory timestamp (fallback: Bonsai workflow launch). **Meaning:** positive values suggest SLAP2 starts first. Error bars represent **assumed uncertainty in Windows clock skew and software delay**, not measured jitter. Use the original timestamps and confidence limits in any presentation.

In [ ]:
sub = df.dropna(subset=["estimated_harp_minus_source_s", "assumed_timing_bound_s"]).copy()
sub = sub.sort_values("session_id")
fig, ax = plt.subplots(figsize=(11, 5))
if len(sub):
    t=sub.estimated_harp_minus_source_s.to_numpy(float)
    u=sub.assumed_timing_bound_s.to_numpy(float)
    ax.errorbar(np.arange(len(sub)), t, yerr=u, fmt="o", ms=3, lw=.7, capsize=1.2)
ax.axhline(0, color="gray", lw=1)
ax.set(xlabel="Sessions (sorted by identifier)", ylabel="HARP start − SLAP2 metadata timestamp (s)", title="Start-order evidence across sessions")
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"02_offset_uncertainty.png", dpi=250);plt.show()

## Figure 3. Start-offset distribution
**Data:** same candidate wall-clock differences as Figure 2. **Meaning:** shows population prevalence of likely SLAP2-first cases, not BonVision–HARP photodiode residuals and not absolute SLAP2–HARP electrical lag.

In [ ]:
fig, ax=plt.subplots(figsize=(9,5))
for category, grp in sub.groupby("order_classification"):
    values=grp.estimated_harp_minus_source_s.dropna()
    if len(values): ax.hist(values, bins=12, alpha=.55, label=category)
ax.axvline(0, color="gray", lw=1);ax.legend()
ax.set(xlabel="HARP minus SLAP2 candidate origin (s)",ylabel="Sessions",title="Estimated start offsets by category")
fig.tight_layout();fig.savefig(OUTPUT_DIR/"03_offset_distribution.png", dpi=250);plt.show()

## Figure 4. The integer-cycle ambiguity
**Data:** time delta divided by the nominal DMD1 cycle period (with integer rounding and conservative bounds). **Meaning:** the number of candidate cycle indices is typically much larger than one. This directly demonstrates why timing based on wall clocks cannot certify one display-frame accuracy.

In [ ]:
sub2=df.dropna(subset=["initial_cycle_offset_estimate", "initial_cycle_offset_lower", "initial_cycle_offset_upper"]).copy()
fig,ax=plt.subplots(figsize=(9,5))
if len(sub2):
    centers=sub2.initial_cycle_offset_estimate.astype(float)
    width=sub2.initial_cycle_offset_upper.astype(float)-sub2.initial_cycle_offset_lower.astype(float)
    ax.scatter(centers, width, alpha=.6)
ax.set(xlabel="Estimated acquisition cycles preceding first HARP pulse",ylabel="Number of candidate cycle indices",title="Absolute alignment uncertainty")
fig.tight_layout();fig.savefig(OUTPUT_DIR/"04_cycle_offset_uncertainty.png",dpi=250);plt.show()

## Table 5. Sessions requiring review before correction
No reprocessing should be launched automatically from these estimates. A reviewed per-epoch integer-cycle offset, its uncertainty, and provenance are required before changing source-to-HARP timing.

In [ ]:
cols=["session_id", "order_classification", "estimated_harp_minus_source_s", "initial_cycle_offset_estimate", "initial_cycle_offset_lower", "initial_cycle_offset_upper", "reprocess_disposition"]
queue=df.loc[df.order_classification.isin(["slap2_first", "ambiguous_overlap", "missing_timing_evidence"]), cols].copy()
display(queue)
queue.to_csv(OUTPUT_DIR/"t03_review_queue.csv",index=False)
print("Review queue:",len(queue),"sessions")

## Interpretation for T0.3

These outputs answer *which start-order scenarios are likely* and how much evidence supports a possible initial source-cycle offset. They **do not** establish exact integer-cycle identity or imaging-to-stimulus precision. For a one-frame acceptance decision, source samples must be mapped to a hardware-verified DI3 cycle index (or a separately validated timing landmark); if no such anchor exists, report `absolute_latency_unverified` and retain the corresponding uncertainty even after exploratory realignment.